In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading songs path data in dict structure

In [2]:
################################################################
#
import os

def load_genre_stems(root_path):

    data = {}

    genres = os.listdir(root_path)

    for genre in genres:

        genre_path = os.path.join(root_path, genre)

        if not os.path.isdir(genre_path):
            continue

        data[genre] = {}

        songs = os.listdir(genre_path)

        for song in songs:

            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            stems = {}

            for file in os.listdir(song_path):

                if file.endswith(".wav"):

                    stem_name = file.replace(".wav","")

                    stems[stem_name] = os.path.join(song_path, file)

            data[genre][song] = stems

    return data

In [3]:
###########################
#
dataset_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

data = load_genre_stems(dataset_path)

In [4]:
#############################
#
print(data["rock"].keys())

dict_keys(['rock.00052', 'rock.00083', 'rock.00092', 'rock.00053', 'rock.00096', 'rock.00003', 'rock.00079', 'rock.00069', 'rock.00097', 'rock.00084', 'rock.00041', 'rock.00012', 'rock.00065', 'rock.00080', 'rock.00086', 'rock.00016', 'rock.00054', 'rock.00062', 'rock.00011', 'rock.00005', 'rock.00038', 'rock.00018', 'rock.00034', 'rock.00022', 'rock.00048', 'rock.00042', 'rock.00088', 'rock.00078', 'rock.00056', 'rock.00006', 'rock.00039', 'rock.00077', 'rock.00028', 'rock.00019', 'rock.00000', 'rock.00074', 'rock.00057', 'rock.00035', 'rock.00007', 'rock.00037', 'rock.00099', 'rock.00075', 'rock.00094', 'rock.00090', 'rock.00093', 'rock.00049', 'rock.00067', 'rock.00024', 'rock.00033', 'rock.00047', 'rock.00085', 'rock.00098', 'rock.00051', 'rock.00032', 'rock.00064', 'rock.00014', 'rock.00066', 'rock.00009', 'rock.00076', 'rock.00020', 'rock.00027', 'rock.00058', 'rock.00044', 'rock.00031', 'rock.00059', 'rock.00025', 'rock.00008', 'rock.00082', 'rock.00063', 'rock.00087', 'rock.000

In [5]:
####################################
#
print(data["rock"]["rock.00004"])

{'drums': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/drums.wav', 'vocals': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/vocals.wav', 'bass': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/bass.wav', 'other': '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00004/other.wav'}


# Loading Noise data paths in dict structurre 

In [6]:
#######################################
#
import pandas as pd
import os

def load_noise_dataset(csv_path, audio_dir):

    df = pd.read_csv(csv_path)

    noise_data = {}

    for _, row in df.iterrows():

        category = row["category"]
        filename = row["filename"]

        path = os.path.join(audio_dir, filename)

        if category not in noise_data:
            noise_data[category] = []

        noise_data[category].append(path)

    return noise_data

In [7]:
#####################################################
#
csv_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/meta/esc50.csv"
audio_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"

noise_data = load_noise_dataset(csv_path, audio_dir)

In [8]:
##################################################
#
print(len(noise_data))
print(noise_data["dog"][:3])

50
['/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-100032-A-0.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-110389-A-0.wav', '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/1-30226-A-0.wav']


In [9]:
##########################################
# checking samipling frequency 
import librosa
# noise_sr = set()

# for category in noise_data:
    
#     for file in noise_data[category]:
        
#         _, sr = librosa.load(file, sr=None)
        
#         noise_sr.add(sr)

# print("Noise sampling rates:", noise_sr)

print("Noise sampling rates: {44100}")

Noise sampling rates: {44100}


In [10]:
################################################
#
# genre_sr = set()

# for genre in data:
    
#     for song in data[genre]:
        
#         for stem in data[genre][song]:
            
#             file = data[genre][song][stem]
            
#             _, sr = librosa.load(file, sr=None)
            
#             genre_sr.add(sr)

# print("Genre sampling rates:", genre_sr)
print("Genre sampling rates: {44100}")

Genre sampling rates: {44100}


# Data Spliting into train and validation 

In [11]:
######################################################
#
import random

def split_dataset(genre_data, train_ratio=0.8):

    train_data = {}
    val_data = {}

    for genre in genre_data:

        songs = list(genre_data[genre].keys())

        random.shuffle(songs)

        split_idx = int(len(songs) * train_ratio)

        train_songs = songs[:split_idx]
        val_songs = songs[split_idx:]

        train_data[genre] = {song: genre_data[genre][song] for song in train_songs}
        val_data[genre] = {song: genre_data[genre][song] for song in val_songs}

    return train_data, val_data


In [12]:
###################################################
#
train_data, val_data = split_dataset(data)
for genre in train_data:
    print(
        genre,
        "train:", len(train_data[genre]),
        "val:", len(val_data[genre])
    )

disco train: 80 val: 20
metal train: 80 val: 20
reggae train: 80 val: 20
blues train: 80 val: 20
rock train: 80 val: 20
classical train: 80 val: 20
jazz train: 80 val: 20
hiphop train: 80 val: 20
country train: 80 val: 20
pop train: 80 val: 20


# Genarating  Traning data 

In [13]:
##################################################
#
import librosa
import numpy as np
import random

SR = 22050
DURATION = 5
SAMPLES = SR * DURATION

In [14]:
#####################################################
#
def load_and_clip(path):

    audio, sr = librosa.load(path, sr=SR)

    if len(audio) > SAMPLES:
        start = random.randint(0, len(audio) - SAMPLES)
        audio = audio[start:start + SAMPLES]

    else:
        pad = SAMPLES - len(audio)
        audio = np.pad(audio, (0, pad))

    return audio

In [15]:
#######################################################
#
def sample_stem_mashup(genre_data, genre):

    songs = random.sample(list(genre_data[genre].keys()), 4)

    stems = {
        "drums":  genre_data[genre][songs[0]]["drums"],
        "bass":   genre_data[genre][songs[1]]["bass"],
        "vocals": genre_data[genre][songs[2]]["vocals"],
        "other": genre_data[genre][songs[3]]["other"]
    }

    return stems

In [16]:
###########################################################
#
def sample_noise(noise_data):

    noise_class = random.choice(list(noise_data.keys()))
    noise_file = random.choice(noise_data[noise_class])

    noise = load_and_clip(noise_file)

    return noise

In [17]:
###########################################################
#
def create_mashup(genre_data, noise_data, genre):

    stems = sample_stem_mashup(genre_data, genre)

    drums = load_and_clip(stems["drums"])
    bass = load_and_clip(stems["bass"])
    vocals = load_and_clip(stems["vocals"])
    others = load_and_clip(stems["other"])

    music_mix = (drums + bass + vocals + others) / 4

    noise = sample_noise(noise_data)

    noise_strength = random.uniform(0.05, 0.2)

    final_audio = music_mix + noise_strength * noise

    final_audio = final_audio / np.max(np.abs(final_audio))

    return final_audio

In [18]:
#############################################################
#
genre = random.choice(list(data.keys()))

audio = create_mashup(data, noise_data, genre)

print(audio.shape)

(110250,)


In [19]:
#############################################################
#
def audio_to_mel(audio):

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=2048,
        hop_length=512,
        n_mels=128
    )

    mel_db = librosa.power_to_db(mel)

    return mel_db

In [20]:
########################################################
#
genres = list(train_data.keys())

genre_to_idx = {g:i for i,g in enumerate(genres)}
idx_to_genre = {i:g for g,i in genre_to_idx.items()}

print(genre_to_idx)

{'disco': 0, 'metal': 1, 'reggae': 2, 'blues': 3, 'rock': 4, 'classical': 5, 'jazz': 6, 'hiphop': 7, 'country': 8, 'pop': 9}


# Data Loaders

In [21]:
###########################################################
#
import torch
from torch.utils.data import Dataset
import random

class MashupDataset(Dataset):

    def __init__(self, genre_data, noise_data):

        self.genre_data = genre_data
        self.noise_data = noise_data
        self.genres = list(genre_data.keys())

    def __len__(self):
        return 2000   # virtual dataset size

    def __getitem__(self, idx):

        genre = random.choice(self.genres)

        audio = create_mashup(self.genre_data, self.noise_data, genre)

        mel = audio_to_mel(audio)

        mel = torch.tensor(mel).unsqueeze(0).float()

        label = genre_to_idx[genre]

        return mel, label


In [22]:
##########################################################
#
from torch.utils.data import DataLoader

train_dataset = MashupDataset(train_data, noise_data)
val_dataset = MashupDataset(val_data, noise_data)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Simple CNN Model

In [23]:
##############################################################
#
import torch.nn as nn
import torch.nn.functional as F

class AudioCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(64*16*27, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x


In [26]:
##############################################
#
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AudioCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(device)

cuda


In [ ]:
#########################################################
#
for epoch in range(10):

    model.train()
    train_loss = 0

    for mel, label in train_loader:

        mel = mel.to(device)
        label = label.to(device)

        pred = model(mel)

        loss = criterion(pred, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print("Epoch:", epoch, "Train Loss:", train_loss/len(train_loader))

In [ ]:
########################################################################
#
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for mel, label in val_loader:

        mel = mel.to(device)
        label = label.to(device)

        pred = model(mel)

        pred_class = torch.argmax(pred, dim=1)

        correct += (pred_class == label).sum().item()

        total += label.size(0)

print("Validation accuracy:", correct/total)

In [ ]:
########################## dummy ##############
data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")
data.to_csv("submission.csv",index=False)